I'm going to now analyse the whole of Layer 2, kernel 12.  
I'm not going to rely on stored stuff right now, its annoying to run and all.  

We'll bascially generate it on the fly and store as pt files.  


- Get samples of saliency maps of l2,k12
- Find the thresholds per point
  - hardcode them
- Find the corresponding patches, might have claude vectorize this maybe (you threshold it, get a mask, get numpy to return non-zero indices. Calculate the patch boundaries, get the patch).  
- Store them as pt files

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    otsu_threshold,
    zeros_with_1_at,
)
from pt_to_api.utils import *
from pt_to_api.capture import get_model_internals
from pt_to_api.mnist import (
    SimpleMNIST,
    get_contribs_for_inp_vectorized,
    get_mnist_dataloader,
)

# Helpers

In [ ]:
import seaborn as sns


def scatter_plot_1d(numbers, suff=""):
    # 2. Create the visualization
    plt.figure(figsize=(20, 3))
    sns.stripplot(x=numbers, color="blue", alpha=0.5, jitter=True)

    plt.title("1D Clustering Visualization" + suff)
    plt.xlabel("Value")
    plt.grid(axis="x", linestyle="--", alpha=0.6)
    plt.show()

# Load model

In [ ]:
MODEL_PATH = Path("../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../pt-to-api/data/first-input-tens.pt")
MAIN_OUT_DIR = (Path.cwd() / "mnist-patches-data")
MAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

dl = get_mnist_dataloader(0.2, bs=256)

# Collection start

In [ ]:
layer_key, channel = "layers.2", 12
input_act_key = "layers.1"
layer = model.get_submodule(layer_key)
device = "mps"
patches_out_dir = MAIN_OUT_DIR / layer_key / str(channel)
patches_out_dir.mkdir(parents=True, exist_ok=True)

subset_size = 10_000

model = model.to(device)

In [ ]:
import torch.nn.functional as F
from torch import nn

# input_act = acts["layers.1"]

def get_indices_of_patches_to_extract(contribs, layer_name, channel, pos_threshes, neg_threshes):
    # print("slice", .shape, "thres", pos_threshes.shape, "neg", neg_threshes.shape)
    contrib_slice = contribs[layer_name][:, channel]
    pos_inds = torch.argwhere(contrib_slice >= pos_threshes)
    neg_inds = torch.argwhere(contrib_slice <= neg_threshes)
    return torch.cat([pos_inds, neg_inds])


def patches_of_single_batch_with_indices(input_act_of_batch, layer, indices):
    op_shape = get_output_shape(
        input_act_of_batch.shape, layer.kernel_size, layer.stride, layer.padding, layer.dilation
    )
    op_r, op_c = op_shape[-2], op_shape[-1]
    b = input_act_of_batch.shape[0]

    patches = F.unfold(
        input_act_of_batch, layer.kernel_size, layer.dilation, layer.padding, layer.stride
    ).reshape(b, -1, op_r, op_c)

    return torch.stack([
        patches[ind[0], :, ind[1], ind[2]]
        for ind in indices
    ])


def get_output_shape(input_shape, ksize, stride, padding, dilation):
    B, C, H, W = input_shape
    Ho = (H + 2*padding[0] - dilation[0]*(ksize[0]-1) - 1) // stride[0] + 1
    Wo = (W + 2*padding[1] - dilation[1]*(ksize[1]-1) - 1) // stride[1] + 1
    return (B, C, Ho, Wo)


# Calculate contribs

In [ ]:
def get_contribs_for_batch(batch, targets, device):
    targs = torch.concat([zeros_with_1_at(10, targ) for targ in targets]).to(device)
    contribs, acts, params = get_contribs_for_inp_vectorized(
        batch, model, targs, "layers.5", device
    )
    return contribs, acts, params

In [ ]:
# we first get all the contribs for some samples
from tqdm import tqdm

collected_contribs = []
for batch, targets in tqdm(dl.train):
    contribs, _, _ = get_contribs_for_batch(batch, targets, device)
    collected_contribs.append(contribs[layer_key][:, channel])
collected_contribs = torch.concat(collected_contribs)

# Visualise

In [ ]:
# we only use the ones with high std and mean
all_stds = []
bad_combs = []

for r in range(collected_contribs.shape[1]):
    for c in range(collected_contribs.shape[2]):
        sm = collected_contribs[:, r, c]
        all_stds.append(sm.std().item())
        if sm.std() < 2e-4 and sm.mean().abs() < 1e-4:
            bad_combs.append((r, c))
plt.plot(all_stds)

# Next steps

In [ ]:
# then otsu threshold, between these steps though
# the user needs to decide which POIs are useful
# this is a subjective test, cuz the attribution method is not very consistent
INF_POS_CONTRIB, INF_NEG_CONTRIB = 2, -2

pos_threshes = np.zeros_like(collected_contribs[0])
neg_threshes = np.zeros_like(collected_contribs[0])


pos_threshes.fill(INF_POS_CONTRIB)
neg_threshes.fill(INF_NEG_CONTRIB)


for r in range(collected_contribs.shape[1]):
    for c in range(collected_contribs.shape[2]):
        if (r, c) in bad_combs:
            print("not relevant", r, c)
        else:
            vals = collected_contribs[:, r, c].numpy()
            pvals = [v for v in vals if v > 0]
            nvals = [v for v in vals if v < 0]
            # scatter_plot_1d(vals)
            plt.show()
            pthres, nthres = otsu_threshold(pvals), otsu_threshold(nvals)
            pos_threshes[r, c] = pthres
            neg_threshes[r, c] = nthres
pos_threshes, neg_threshes = torch.tensor(pos_threshes), torch.tensor(neg_threshes)

In [ ]:
import gc
gc.collect()

In [ ]:
torch.save(pos_threshes, Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/12/pos_threshes.pt")
torch.save(neg_threshes, Path.home() / "Desktop/gdrive-sync/l2-o12-patches/cosine-anneal/comps/12/neg_threshes.pt")

In [ ]:
main_dl = get_mnist_dataloader(1, bs=1024)

In [ ]:
# patches_out_dir

for i, (batch, targets) in enumerate(tqdm(main_dl.train)):
    contribs, acts, params = get_contribs_for_batch(batch, targets, device)
    indices = get_indices_of_patches_to_extract(contribs, layer_key, channel, pos_threshes, neg_threshes)
    patches_of_batch = patches_of_single_batch_with_indices(acts[input_act_key], layer, indices)
    torch.save(patches_of_batch, patches_out_dir / f"{i}.pt")